# 📚 내 기록 → 마크다운 변환기

흩어져 있는 내 지식·경험을 **AI가 읽기 좋은 마크다운(.md)** 으로 모읍니다.

> PKEMS(개인지식경험관리체계) 프로젝트

### 이 노트북으로 할 수 있는 것

| | 무엇을 | 어떤 형식 |
|---|---|---|
| **1부** | 네이버 블로그 백업 | `.pdf` (글마다 나눠서 변환) |
| **2부** | 문서 폴더 통째로 | `.hwp` `.hwpx` `.docx` `.pptx` `.xlsx` `.pdf` `.html` `.txt` |
| **3부** | 구글 문서 | 구글 문서·시트·슬라이드 |

---

### 사용 방법

**먼저 아래 '준비하기' 두 칸을 실행**한 뒤, 필요한 부(1·2·3)로 가서
각 칸의 **▶ 버튼**을 순서대로 누르면 됩니다.

- 중간에 끊겨도 다시 누르면 **이어서** 진행됩니다
- 한 파일이 실패해도 나머지는 계속 변환됩니다
- 모든 작업은 **본인 구글 드라이브 안에서만** 이루어집니다

⏱️ 파일 100MB당 대략 1~3분.

## 🔧 준비하기 (설치 + 구글 드라이브 연결) — 맨 처음 한 번

▶ 를 누르면 구글 계정 접근 허용을 물어봅니다. **허용**을 눌러주세요.
내 드라이브 안에서만 작업하며, 파일이 외부로 나가지 않습니다.

In [ ]:
#@title ▶ 눌러서 준비하기 { display-mode: "form" }
import subprocess, sys

print("① 필요한 프로그램 설치 중... (30초쯤 걸립니다)")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pymupdf",          # PDF
                "olefile",          # 한글 .hwp
                "python-docx",      # 워드
                "python-pptx",      # 파워포인트
                "openpyxl",         # 엑셀
                "beautifulsoup4",   # HTML
                ], check=False)

print("② 구글 드라이브 연결 중...")
from google.colab import drive
drive.mount('/content/drive')

print("\n준비 완료! 다음 칸으로 넘어가세요.")

### (자동) 변환 엔진 불러오기 — 이 칸도 ▶ 눌러주세요

In [ ]:
#@title ▶ 눌러서 엔진 불러오기 { display-mode: "form" }
import base64, pathlib, importlib, sys

_ENGINES = {
  "pkems_converter.py": (
    "..."
  ),
  "pkems_readers.py": (
    "..."
  ),
  "pkems_privacy.py": (
    "..."
    "dH3qsbQiKQogICAgcmV0dXJuIGhpdHMK"
  ),
  "pkems_folder.py": (
    "IyAtKi0gY29kaW5nOiB1dGYtOCAtKi0KIiIiClBLRU1TIO2PtOuNlCDsnbzqtIQg67OA7ZmY6riwCj09PT09PT09PT09PT09PT09"
    "PT09PT09Cu2PtOuNlCDtlZjrgpjrpbwg7Ya17Ke466GcIO2bkeyWtOyEnCwg7JWI7JeQIOyeiOuKlCDrqqjrk6Ag66y47ISc66W8"
    "IOuniO2BrOuLpOyatCgubWQp7Jy866GcIOuwlOq+vOuLpC4K7ZWc6riAKC5od3AvLmh3cHgpLCDsm4zrk5wsIO2MjOybjO2PrOyd"
    "uO2KuCwg7JeR7IWALCBQREYsIEhUTUwsIO2FjeyKpO2KuOulvCDrqqjrkZAg64uk66Os64ukLgoKICAgIGZyb20gcGtlbXNfZm9s"
    "ZGVyIGltcG9ydCBGb2xkZXJDb252ZXJ0ZXIsIEZvbGRlclNldHRpbmdzCgogICAgZmMgPSBGb2xkZXJDb252ZXJ0ZXIoRm9sZGVy"
    "U2V0dGluZ3MoCiAgICAgICAgc3JjX2RpciA9ICIvY29udGVudC9kcml2ZS9NeURyaXZlLzAxX+2Vmeq1kCIsCiAgICAgICAgb3V0"
    "X2RpciA9ICIvY29udGVudC9kcml2ZS9NeURyaXZlL1BLRU1TL+uzgO2ZmOqysOqzvCIsCiAgICApKQogICAgZmMuc2NhbigpICAg"
    "ICAgIyDrqLzsoIAg66y07JeH7J20IOuqhyDqsJwg7J6I64qU7KeAIO2ZleyduAogICAgZmMucnVuKCkgICAgICAgIyDrs4DtmZgK"
    "Cu2KueynlQogICAgLSDsm5Drnpgg7Y+0642UIOq1rOyhsOulvCDqt7jrjIDroZwg7Jyg7KeA7ZWc64ukCiAgICAtIOydtOuvuCDr"
    "s4DtmZjtlZwg7YyM7J287J2AIOqxtOuEiOubtOuLpCAo7KSR6rCE7JeQIOuBiuqyqOuPhCDsnbTslrTshJwg7KeE7ZaJKQogICAg"
    "LSDtlZwg7YyM7J287J20IOyLpO2MqO2VtOuPhCDsoITssrTqsIAg66mI7LaU7KeAIOyViuuKlOuLpCAo7Jik66WY64qUIOuUsOuh"
    "nCDquLDroZ0pCiAgICAtIOuzgO2ZmCDqsrDqs7wg66qp66GdKElOREVYLm1kLCBfZmlsZXMuanNvbinsnYQg66eM65Og64ukCgpQ"
    "S0VNUyjqsJzsnbjsp4Dsi53qsr3tl5jqtIDrpqzssrTqs4QpIO2UhOuhnOygne2KuAoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBv"
    "cnQgYW5ub3RhdGlvbnMKCmltcG9ydCBvcwppbXBvcnQgaW8KaW1wb3J0IHJlCmltcG9ydCBqc29uCmltcG9ydCB0aW1lCmltcG9y"
    "dCBjb2xsZWN0aW9ucwpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCgpmcm9tIHBrZW1zX3JlYWRlcnMg"
    "aW1wb3J0IHJlYWRfYW55LCBSRUFERVJTLCBSZWFkUmVzdWx0CmZyb20gcGtlbXNfcHJpdmFjeSBpbXBvcnQgUHJpdmFjeUZpbHRl"
    "ciwgUG9saWN5LCBQcml2YWN5UmVwb3J0CgoKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi"
    "lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi"
    "lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKQGRhdGFjbGFzcwpj"
    "bGFzcyBGb2xkZXJTZXR0aW5nczoKICAgIHNyY19kaXI6IHN0ciAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyDt"
    "m5HsnYQg7JuQ67O4IO2PtOuNlAogICAgb3V0X2Rpcjogc3RyID0gIiIgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIOqy"
    "sOqzvCDtj7TrjZQgKOu5hOyasOuptCBzcmNfZGlyL19tZCkKICAgIGluY2x1ZGU6IHR1cGxlID0gdHVwbGUoUkVBREVSUykgICAg"
    "ICAgICAgICAgICAgIyDri6Tro7Ag7ZmV7J6l7J6QCiAgICBleGNsdWRlX2RpcnM6IHR1cGxlID0gKCIuZ2l0IiwgIi5vYnNpZGlh"
    "biIsICJfX3B5Y2FjaGVfXyIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICJub2RlX21vZHVsZXMiLCAiaW1hZ2VzIiwgIl9t"
    "ZCIpCiAgICBza2lwX2V4aXN0aW5nOiBib29sID0gVHJ1ZSAgICAgICAgICAgICAgICAgICAgICMg7J2066+4IOyeiOuKlCBtZCDr"
    "ipQg6rG064SI65uw6riwCiAgICBtaW5fY2hhcnM6IGludCA9IDEwICAgICAgICAgICAgICAgICAgICAgICAgICAgICMg7J2067O0"
    "64ukIOynp+ycvOuptCAn64K07JqpIOyXhuydjCfsnLzroZwg6riw66GdCiAgICBtYXhfbWI6IGZsb2F0ID0gMjAwLjAgICAgICAg"
    "ICAgICAgICAgICAgICAgICAgICMg7J2067O064ukIO2BsCDtjIzsnbzsnYAg6rG064SI65uw6riwCiAgICBrZWVwX3RyZWU6IGJv"
    "b2wgPSBUcnVlICAgICAgICAgICAgICAgICAgICAgICAgICMg7JuQ67O4IO2PtOuNlCDqtazsobAg7Jyg7KeACiAgICB2ZXJib3Nl"
    "OiBib29sID0gVHJ1ZQoKICAgICMg4pSA4pSAIOqwnOyduOygleuztCDsspjrpqwg4pSA4pSACiAgICDqsJzsnbjsoJXrs7Rf6rCA"
    "66as6riwOiBib29sID0gVHJ1ZSAgICAgICAgICAgICAgICAgICAgIyDrgYTrqbQg7JuQ66y4IOq3uOuMgOuhnCDsoIDsnqUKICAg"
    "IOqwnOyduOygleuztF/soJXssYU6IFBvbGljeSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1Qb2xpY3kpCiAgICDrs7Tqs6DshJxf"
    "7JuQ67O47ZGc7IucOiBib29sID0gVHJ1ZSAgICAgICAgICAgICAgICAgICAgIyDrs7Tqs6DshJzsl5Ag6rCA66as6riwIOyghCDq"
    "sJLsnYQg64Ko6ri47KeACgogICAgZGVmIHJlc29sdmVkX291dChzZWxmKSAtPiBzdHI6CiAgICAgICAgcmV0dXJuIHNlbGYub3V0"
    "X2RpciBvciBvcy5wYXRoLmpvaW4oc2VsZi5zcmNfZGlyLCAiX21kIikKCgpfSU5WQUxJRCA9IHJlLmNvbXBpbGUocidbXFwvOio/"
    "Ijw+fF0nKQoKCmRlZiBzYWZlX25hbWUobmFtZTogc3RyLCBtYXhsZW46IGludCA9IDkwKSAtPiBzdHI6CiAgICBzID0gX0lOVkFM"
    "SUQuc3ViKCJfIiwgbmFtZSkuc3RyaXAoKQogICAgcmV0dXJuIHNbOm1heGxlbl0ucnN0cmlwKCIgLiIpIG9yICLrrLTsoJwiCgoK"
    "IyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi"
    "lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi"
    "lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKY2xhc3MgRm9sZGVyQ29udmVydGVyOgogICAgZGVmIF9faW5pdF9f"
    "KHNlbGYsIHNldHRpbmdzOiBGb2xkZXJTZXR0aW5ncyk6CiAgICAgICAgc2VsZi5zID0gc2V0dGluZ3MKICAgICAgICBzZWxmLm91"
    "dCA9IHNldHRpbmdzLnJlc29sdmVkX291dCgpCiAgICAgICAgc2VsZi5yZWNvcmRzOiBsaXN0W2RpY3RdID0gW10KICAgICAgICBz"
    "ZWxmLmVycm9yczogbGlzdFtkaWN0XSA9IFtdCiAgICAgICAgc2VsZi5zdGF0cyA9IGNvbGxlY3Rpb25zLkNvdW50ZXIoKQogICAg"
    "ICAgIHNlbGYucHJpdmFjeSA9IFByaXZhY3lGaWx0ZXIoc2V0dGluZ3Mu6rCc7J247KCV67O0X+ygleyxhSkgXAogICAgICAgICAg"
    "ICBpZiBzZXR0aW5ncy7qsJzsnbjsoJXrs7Rf6rCA66as6riwIGVsc2UgTm9uZQogICAgICAgIHNlbGYucmVwb3J0ID0gUHJpdmFj"
    "eVJlcG9ydChzaG93X29yaWdpbmFsPXNldHRpbmdzLuuztOqzoOyEnF/sm5Drs7jtkZzsi5wpCgogICAgZGVmIGxvZyhzZWxmLCAq"
    "YSk6CiAgICAgICAgaWYgc2VsZi5zLnZlcmJvc2U6CiAgICAgICAgICAgIHByaW50KCphLCBmbHVzaD1UcnVlKQoKICAgICMg4pSA"
    "4pSAIOuMgOyDgSDtjIzsnbwg7IiY7KeRCiAgICBkZWYgY29sbGVjdChzZWxmKSAtPiBsaXN0W3N0cl06CiAgICAgICAgZm91bmQg"
    "PSBbXQogICAgICAgIGV4dHMgPSB7ZS5sb3dlcigpIGZvciBlIGluIHNlbGYucy5pbmNsdWRlfQogICAgICAgIG91dF9hYnMgPSBv"
    "cy5wYXRoLmFic3BhdGgoc2VsZi5vdXQpCiAgICAgICAgZm9yIGRwLCBkbnMsIGZucyBpbiBvcy53YWxrKHNlbGYucy5zcmNfZGly"
    "KToKICAgICAgICAgICAgZG5zWzpdID0gW2QgZm9yIGQgaW4gZG5zCiAgICAgICAgICAgICAgICAgICAgICBpZiBkIG5vdCBpbiBz"
    "ZWxmLnMuZXhjbHVkZV9kaXJzIGFuZCBub3QgZC5zdGFydHN3aXRoKCIuIildCiAgICAgICAgICAgIGlmIG9zLnBhdGguYWJzcGF0"
    "aChkcCkuc3RhcnRzd2l0aChvdXRfYWJzKToKICAgICAgICAgICAgICAgIGNvbnRpbnVlICAgICAgICAgICAgICAgICAgICAgICAg"
    "ICAgICAgICAgICAjIOqysOqzvCDtj7TrjZTripQg7KCc7Jm4CiAgICAgICAgICAgIGZvciBmbiBpbiBmbnM6CiAgICAgICAgICAg"
    "ICAgICBpZiBmbi5zdGFydHN3aXRoKCJ+JCIpIG9yIGZuLnN0YXJ0c3dpdGgoIi4iKToKICAgICAgICAgICAgICAgICAgICBjb250"
    "aW51ZQogICAgICAgICAgICAgICAgaWYgb3MucGF0aC5zcGxpdGV4dChmbilbMV0ubG93ZXIoKSBpbiBleHRzOgogICAgICAgICAg"
    "ICAgICAgICAgIGZvdW5kLmFwcGVuZChvcy5wYXRoLmpvaW4oZHAsIGZuKSkKICAgICAgICByZXR1cm4gc29ydGVkKGZvdW5kKQoK"
    "ICAgICMg4pSA4pSAIO2bkeyWtOuztOq4sCAo67OA7ZmYIOyXhuydtCDtmITtmanrp4wpCiAgICBkZWYgc2NhbihzZWxmKSAtPiBk"
    "aWN0OgogICAgICAgIGZpbGVzID0gc2VsZi5jb2xsZWN0KCkKICAgICAgICBieV9leHQgPSBjb2xsZWN0aW9ucy5Db3VudGVyKG9z"
    "LnBhdGguc3BsaXRleHQoZilbMV0ubG93ZXIoKSBmb3IgZiBpbiBmaWxlcykKICAgICAgICB0b3RhbF9tYiA9IDAuMAogICAgICAg"
    "IGZvciBmIGluIGZpbGVzOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICB0b3RhbF9tYiArPSBvcy5wYXRoLmdldHNp"
    "emUoZikgLyAxMDI0IC8gMTAyNAogICAgICAgICAgICBleGNlcHQgT1NFcnJvcjoKICAgICAgICAgICAgICAgIHBhc3MKCiAgICAg"
    "ICAgc2VsZi5sb2coZiLrjIDsg4Eg7Y+0642UIDoge3NlbGYucy5zcmNfZGlyfSIpCiAgICAgICAgc2VsZi5sb2coZiLssL7snYAg"
    "7YyM7J28IDoge2xlbihmaWxlcyl96rCcICh7dG90YWxfbWI6LjBmfSBNQilcbiIpCiAgICAgICAgc2VsZi5sb2coIiAg7ZiV7Iud"
    "67OEIikKICAgICAgICBmb3IgZSwgbiBpbiBieV9leHQubW9zdF9jb21tb24oKToKICAgICAgICAgICAgc2VsZi5sb2coZiIgICAg"
    "e2U6OH0ge246NX3qsJwiKQogICAgICAgIHNlbGYubG9nKGYiXG4gIOyggOyepSDsnITsuZggOiB7c2VsZi5vdXR9IikKICAgICAg"
    "ICByZXR1cm4geyJmaWxlcyI6IGxlbihmaWxlcyksICJieV9leHQiOiBkaWN0KGJ5X2V4dCksICJtYiI6IHJvdW5kKHRvdGFsX21i"
    "KX0KCiAgICAjIOKUgOKUgCDqsrDqs7wgbWQg6rK966GcIOygle2VmOq4sAogICAgZGVmIG1kX3BhdGhfZm9yKHNlbGYsIHNyYzog"
    "c3RyKSAtPiBzdHI6CiAgICAgICAgcmVsID0gb3MucGF0aC5yZWxwYXRoKHNyYywgc2VsZi5zLnNyY19kaXIpCiAgICAgICAgaGVh"
    "ZCwgZm4gPSBvcy5wYXRoLnNwbGl0KHJlbCkKICAgICAgICBzdGVtLCBleHQgPSBvcy5wYXRoLnNwbGl0ZXh0KGZuKQogICAgICAg"
    "IG5hbWUgPSBzYWZlX25hbWUoZiJ7c3RlbX17ZXh0LnJlcGxhY2UoJy4nLCAnXycpfSIpICsgIi5tZCIKICAgICAgICBpZiBzZWxm"
    "LnMua2VlcF90cmVlIGFuZCBoZWFkIGFuZCBoZWFkICE9ICIuIjoKICAgICAgICAgICAgaGVhZCA9IG9zLnBhdGguam9pbigqW3Nh"
    "ZmVfbmFtZShwKSBmb3IgcCBpbiBoZWFkLnNwbGl0KG9zLnNlcCldKQogICAgICAgICAgICByZXR1cm4gb3MucGF0aC5qb2luKHNl"
    "bGYub3V0LCBoZWFkLCBuYW1lKQogICAgICAgIHJldHVybiBvcy5wYXRoLmpvaW4oc2VsZi5vdXQsIG5hbWUpCgogICAgIyDilIDi"
    "lIAg66i466as66eQIOunjOuTpOq4sAogICAgZGVmIGZyb250X21hdHRlcihzZWxmLCBzcmM6IHN0ciwgcmVzOiBSZWFkUmVzdWx0"
    "KSAtPiBzdHI6CiAgICAgICAgc3QgPSBvcy5zdGF0KHNyYykKICAgICAgICBtdGltZSA9IHRpbWUuc3RyZnRpbWUoIiVZLSVtLSVk"
    "ICVIOiVNIiwgdGltZS5sb2NhbHRpbWUoc3Quc3RfbXRpbWUpKQogICAgICAgIHJlbCA9IG9zLnBhdGgucmVscGF0aChzcmMsIHNl"
    "bGYucy5zcmNfZGlyKS5yZXBsYWNlKCJcXCIsICIvIikKICAgICAgICB0aXRsZSA9IG9zLnBhdGguc3BsaXRleHQob3MucGF0aC5i"
    "YXNlbmFtZShzcmMpKVswXS5yZXBsYWNlKCciJywgIiciKQogICAgICAgIGV4dHJhID0gIiIKICAgICAgICBpZiByZXMubWV0YToK"
    "ICAgICAgICAgICAgYml0cyA9ICIgwrcgIi5qb2luKGYie2t9IHt2fSIgZm9yIGssIHYgaW4gcmVzLm1ldGEuaXRlbXMoKQogICAg"
    "ICAgICAgICAgICAgICAgICAgICAgICAgICBpZiBrIG5vdCBpbiAoInVybCIsICJkb2NfaWQiKSkKICAgICAgICAgICAgaWYgYml0"
    "czoKICAgICAgICAgICAgICAgIGV4dHJhID0gZiJpbmZvOiB7Yml0c31cbiIKICAgICAgICByZXR1cm4gKAogICAgICAgICAgICAi"
    "LS0tXG4iCiAgICAgICAgICAgIGYndGl0bGU6ICJ7dGl0bGV9IlxuJwogICAgICAgICAgICBmImRhdGU6IHttdGltZX1cbiIKICAg"
    "ICAgICAgICAgZidzb3VyY2U6ICJ7cmVsfSJcbicKICAgICAgICAgICAgZidraW5kOiAie3Jlcy5raW5kfSJcbicKICAgICAgICAg"
    "ICAgZiJ7ZXh0cmF9IgogICAgICAgICAgICAiLS0tXG5cbiIKICAgICAgICAgICAgZiIjIHt0aXRsZX1cblxuIgogICAgICAgICAg"
    "ICBmIirsm5Drs7g6IGB7cmVsfWAgwrcg7IiY7KCVIHttdGltZX0qXG5cbiIKICAgICAgICApCgogICAgIyDilIDilIAg7Iuk7ZaJ"
    "CiAgICBkZWYgcnVuKHNlbGYsIGxpbWl0OiBpbnQgfCBOb25lID0gTm9uZSkgLT4gZGljdDoKICAgICAgICB0MCA9IHRpbWUudGlt"
    "ZSgpCiAgICAgICAgb3MubWFrZWRpcnMoc2VsZi5vdXQsIGV4aXN0X29rPVRydWUpCiAgICAgICAgZmlsZXMgPSBzZWxmLmNvbGxl"
    "Y3QoKQogICAgICAgIGlmIGxpbWl0OgogICAgICAgICAgICBmaWxlcyA9IGZpbGVzWzpsaW1pdF0KICAgICAgICB0b3RhbCA9IGxl"
    "bihmaWxlcykKICAgICAgICBzZWxmLmxvZyhmIu2MjOydvCB7dG90YWx96rCc66W8IOuzgO2ZmO2VqeuLiOuLpC5cbiIpCgogICAg"
    "ICAgIGRvbmUgPSBza2lwcGVkID0gZmFpbGVkID0gMAogICAgICAgIGZvciBpLCBzcmMgaW4gZW51bWVyYXRlKGZpbGVzLCAxKToK"
    "ICAgICAgICAgICAgZHN0ID0gc2VsZi5tZF9wYXRoX2ZvcihzcmMpCgogICAgICAgICAgICBpZiBzZWxmLnMuc2tpcF9leGlzdGlu"
    "ZyBhbmQgb3MucGF0aC5leGlzdHMoZHN0KToKICAgICAgICAgICAgICAgIHNraXBwZWQgKz0gMQogICAgICAgICAgICAgICAgY29u"
    "dGludWUKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgbWIgPSBvcy5wYXRoLmdldHNpemUoc3JjKSAvIDEwMjQgLyAx"
    "MDI0CiAgICAgICAgICAgIGV4Y2VwdCBPU0Vycm9yOgogICAgICAgICAgICAgICAgbWIgPSAwCiAgICAgICAgICAgIGlmIG1iID4g"
    "c2VsZi5zLm1heF9tYjoKICAgICAgICAgICAgICAgIHNlbGYuZXJyb3JzLmFwcGVuZCh7ImZpbGUiOiBzcmMsICJlcnJvciI6IGYi"
    "64SI66y0IO2BvCAoe21iOi4wZn1NQikifSkKICAgICAgICAgICAgICAgIGZhaWxlZCArPSAxCiAgICAgICAgICAgICAgICBjb250"
    "aW51ZQoKICAgICAgICAgICAgcmVzID0gcmVhZF9hbnkoc3JjKQogICAgICAgICAgICBpZiBub3QgcmVzLm9rOgogICAgICAgICAg"
    "ICAgICAgc2VsZi5lcnJvcnMuYXBwZW5kKHsiZmlsZSI6IHNyYywgImVycm9yIjogcmVzLmVycm9yfSkKICAgICAgICAgICAgICAg"
    "IHNlbGYuc3RhdHNbZiLsi6TtjKg6e3Jlcy5raW5kfSJdICs9IDEKICAgICAgICAgICAgICAgIGZhaWxlZCArPSAxCiAgICAgICAg"
    "ICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgYm9keSA9IHJlcy50ZXh0CiAgICAgICAgICAgIHJlbCA9IG9zLnBhdGgucmVs"
    "cGF0aChzcmMsIHNlbGYucy5zcmNfZGlyKS5yZXBsYWNlKCJcXCIsICIvIikKCiAgICAgICAgICAgIOqwnOyduOygleuztF/qsbTs"
    "iJggPSAwCiAgICAgICAgICAgIGlmIHNlbGYucHJpdmFjeSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIGJvZHksIGhpdHMg"
    "PSBzZWxmLnByaXZhY3kubWFzayhib2R5KQogICAgICAgICAgICAgICAgaWYgaGl0czoKICAgICAgICAgICAgICAgICAgICBzZWxm"
    "LnJlcG9ydC5hZGQocmVsLCBoaXRzKQogICAgICAgICAgICAgICAgICAgIOqwnOyduOygleuztF/qsbTsiJggPSBsZW4oaGl0cykK"
    "ICAgICAgICAgICAgICAgICAgICBzZWxmLnN0YXRzWyLqsJzsnbjsoJXrs7TqsIDrprwiXSArPSBsZW4oaGl0cykKCiAgICAgICAg"
    "ICAgIG5vdGUgPSAiIgogICAgICAgICAgICBpZiBsZW4oYm9keSkgPCBzZWxmLnMubWluX2NoYXJzOgogICAgICAgICAgICAgICAg"
    "bm90ZSA9ICgiXG4+IOKaoCDquIDsnpDrpbwg6rGw7J2YIOywvuyngCDrqrvtlojsirXri4jri6QuIOq3uOumvCDsnITso7zsnbTq"
    "sbDrgpgg7Iqk7LqU7ZWcIOusuOyEnOydvCDsiJggIgogICAgICAgICAgICAgICAgICAgICAgICAi7J6I7Iq164uI64ukLiDsnbQg"
    "64+E6rWs64qUIOq4gOyekCDsnbjsi50oT0NSKeydhCDtlZjsp4Ag7JWK7Iq164uI64ukLlxuIikKICAgICAgICAgICAgICAgIHNl"
    "bGYuc3RhdHNbIuuCtOyaqeqxsOydmOyXhuydjCJdICs9IDEKCiAgICAgICAgICAgIG9zLm1ha2VkaXJzKG9zLnBhdGguZGlybmFt"
    "ZShkc3QpLCBleGlzdF9vaz1UcnVlKQogICAgICAgICAgICB3aXRoIGlvLm9wZW4oZHN0LCAidyIsIGVuY29kaW5nPSJ1dGYtOCIp"
    "IGFzIGY6CiAgICAgICAgICAgICAgICBmLndyaXRlKHNlbGYuZnJvbnRfbWF0dGVyKHNyYywgcmVzKSArIG5vdGUgKyBib2R5ICsg"
    "IlxuIikKCiAgICAgICAgICAgIHNlbGYucmVjb3Jkcy5hcHBlbmQoewogICAgICAgICAgICAgICAgInRpdGxlIjogb3MucGF0aC5z"
    "cGxpdGV4dChvcy5wYXRoLmJhc2VuYW1lKHNyYykpWzBdLAogICAgICAgICAgICAgICAgImtpbmQiOiByZXMua2luZCwKICAgICAg"
    "ICAgICAgICAgICJjaGFycyI6IHJlcy5jaGFycywKICAgICAgICAgICAgICAgICJzcmMiOiByZWwsCiAgICAgICAgICAgICAgICAi"
    "bWQiOiBvcy5wYXRoLnJlbHBhdGgoZHN0LCBzZWxmLm91dCkucmVwbGFjZSgiXFwiLCAiLyIpLAogICAgICAgICAgICAgICAgImRh"
    "dGUiOiB0aW1lLnN0cmZ0aW1lKCIlWS0lbS0lZCIsIHRpbWUubG9jYWx0aW1lKG9zLnN0YXQoc3JjKS5zdF9tdGltZSkpLAogICAg"
    "ICAgICAgICAgICAgIuqwnOyduOygleuztCI6IOqwnOyduOygleuztF/qsbTsiJgsCiAgICAgICAgICAgIH0pCiAgICAgICAgICAg"
    "IHNlbGYuc3RhdHNbcmVzLmtpbmRdICs9IDEKICAgICAgICAgICAgZG9uZSArPSAxCgogICAgICAgICAgICBpZiBkb25lIGFuZCBk"
    "b25lICUgNTAgPT0gMDoKICAgICAgICAgICAgICAgIHNlbGYubG9nKGYiICDigKYge2RvbmV96rCcIOuzgO2ZmCAoe2l9L3t0b3Rh"
    "bH0pIikKCiAgICAgICAgc2VsZi53cml0ZV9pbmRleCgpCiAgICAgICAg67O06rOg7IScID0gc2VsZi5yZXBvcnQud3JpdGUoc2Vs"
    "Zi5vdXQpIGlmIHNlbGYucHJpdmFjeSBpcyBub3QgTm9uZSBlbHNlIE5vbmUKCiAgICAgICAgc2VjcyA9IGludCh0aW1lLnRpbWUo"
    "KSAtIHQwKQogICAgICAgIHNlbGYubG9nKGYiXG7smYTro4whIOuzgO2ZmCB7ZG9uZX3qsJwgwrcg6rG064SI65yAIHtza2lwcGVk"
    "feqwnCDCtyDsi6TtjKgge2ZhaWxlZH3qsJwgIgogICAgICAgICAgICAgICAgIGYiwrcge3NlY3MgLy8gNjB967aEIHtzZWNzICUg"
    "NjB97LSIIikKICAgICAgICBpZiBzZWxmLnN0YXRzOgogICAgICAgICAgICBzZWxmLmxvZygiXG4gIO2YleyLneuzhCDqsrDqs7wi"
    "KQogICAgICAgICAgICBmb3IgaywgbiBpbiBzZWxmLnN0YXRzLm1vc3RfY29tbW9uKCk6CiAgICAgICAgICAgICAgICBzZWxmLmxv"
    "ZyhmIiAgICB7azoxNH0ge246NX3qsJwiKQogICAgICAgIGlmIHNlbGYuZXJyb3JzOgogICAgICAgICAgICBzZWxmLmxvZyhmIlxu"
    "ICDsi6TtjKgg66qp66GdIDoge29zLnBhdGguam9pbihzZWxmLm91dCwgJ1/smKTrpZgubWQnKX0iKQoKICAgICAgICBpZiBzZWxm"
    "LnByaXZhY3kgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHNlbGYubG9nKCJcbiIgKyAi4pSAIiAqIDQ2KQogICAgICAgICAgICBz"
    "ZWxmLmxvZyhzZWxmLnJlcG9ydC5zdW1tYXJ5KCkpCiAgICAgICAgICAgIGlmIOuztOqzoOyEnDoKICAgICAgICAgICAgICAgIHNl"
    "bGYubG9nKGYiXG4gIOyekOyEuO2VnCDrgrTsl60gOiB767O06rOg7IScfSIpCiAgICAgICAgICAgICAgICBpZiBzZWxmLnMu67O0"
    "6rOg7IScX+ybkOuzuO2RnOyLnDoKICAgICAgICAgICAgICAgICAgICBzZWxmLmxvZygiICDimqAg7J20IOuztOqzoOyEnOyXkOuK"
    "lCDqsIDrpqzquLAg7KCEIOybkOuzuOydtCDrk6TslrQg7J6I7Iq164uI64ukLiDqs7XsnKDtlZjsp4Ag66eI7IS47JqULiIpCgog"
    "ICAgICAgIHJldHVybiB7ImRvbmUiOiBkb25lLCAic2tpcHBlZCI6IHNraXBwZWQsICJmYWlsZWQiOiBmYWlsZWQsCiAgICAgICAg"
    "ICAgICAgICAic3RhdHMiOiBkaWN0KHNlbGYuc3RhdHMpLAogICAgICAgICAgICAgICAgIuqwnOyduOygleuztCI6IGxlbihzZWxm"
    "LnJlcG9ydC5yb3dzKX0KCiAgICAjIOKUgOKUgCDrqqnssKgv7Jik66WYIOq4sOuhnQogICAgZGVmIHdyaXRlX2luZGV4KHNlbGYp"
    "OgogICAgICAgIGlmIHNlbGYucmVjb3JkczoKICAgICAgICAgICAgd2l0aCBpby5vcGVuKG9zLnBhdGguam9pbihzZWxmLm91dCwg"
    "Il9maWxlcy5qc29uIiksICJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICAgICAgICAgIGpzb24uZHVtcChzZWxm"
    "LnJlY29yZHMsIGYsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTEpCgogICAgICAgICAgICBieV9raW5kID0gY29sbGVjdGlv"
    "bnMuZGVmYXVsdGRpY3QobGlzdCkKICAgICAgICAgICAgZm9yIHIgaW4gc2VsZi5yZWNvcmRzOgogICAgICAgICAgICAgICAgYnlf"
    "a2luZFtyWyJraW5kIl1dLmFwcGVuZChyKQoKICAgICAgICAgICAgTCA9IFsiIyDwn5OCIOuzgO2ZmOuQnCDrrLjshJwg66qp7LCo"
    "IiwgIiIsCiAgICAgICAgICAgICAgICAgZiLsoITssrQgKip7bGVuKHNlbGYucmVjb3Jkcyl96rCcKiogwrcg7J6Q64+ZIOyDneyE"
    "sSIsICIiLAogICAgICAgICAgICAgICAgICJ8IO2YleyLnSB8IOqwnOyImCB8IiwgInwtLS0tLS18LS0tLS06fCJdCiAgICAgICAg"
    "ICAgIGZvciBrIGluIHNvcnRlZChieV9raW5kLCBrZXk9bGFtYmRhIHg6IC1sZW4oYnlfa2luZFt4XSkpOgogICAgICAgICAgICAg"
    "ICAgTC5hcHBlbmQoZiJ8IHtrfSB8IHtsZW4oYnlfa2luZFtrXSl9IHwiKQogICAgICAgICAgICBMLmFwcGVuZCgiIikKICAgICAg"
    "ICAgICAgZm9yIGsgaW4gc29ydGVkKGJ5X2tpbmQsIGtleT1sYW1iZGEgeDogLWxlbihieV9raW5kW3hdKSk6CiAgICAgICAgICAg"
    "ICAgICBMICs9IFtmIiMjIHtrfSAoe2xlbihieV9raW5kW2tdKX3qsJwpIiwgIiJdCiAgICAgICAgICAgICAgICBmb3IgciBpbiBz"
    "b3J0ZWQoYnlfa2luZFtrXSwga2V5PWxhbWJkYSB4OiB4WyJzcmMiXSk6CiAgICAgICAgICAgICAgICAgICAgTC5hcHBlbmQoZiIt"
    "IFt7clsndGl0bGUnXX1dKHtyWydtZCddfSkgIgogICAgICAgICAgICAgICAgICAgICAgICAgICAgIGYiwrcge3JbJ2NoYXJzJ106"
    "LH3snpAgwrcgYHtyWydzcmMnXX1gIikKICAgICAgICAgICAgICAgIEwuYXBwZW5kKCIiKQogICAgICAgICAgICB3aXRoIGlvLm9w"
    "ZW4ob3MucGF0aC5qb2luKHNlbGYub3V0LCAiSU5ERVgubWQiKSwgInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAg"
    "ICAgICAgICAgZi53cml0ZSgiXG4iLmpvaW4oTCkpCgogICAgICAgIGlmIHNlbGYuZXJyb3JzOgogICAgICAgICAgICBMID0gWyIj"
    "IOKaoCDrs4DtmZjtlZjsp4Ag66q77ZWcIO2MjOydvCIsICIiLAogICAgICAgICAgICAgICAgIGYie2xlbihzZWxmLmVycm9ycyl9"
    "6rCcIiwgIiIsCiAgICAgICAgICAgICAgICAgInwg7YyM7J28IHwg7J207Jyg7JmAIO2VtOqysCDrsKnrspUgfCIsICJ8LS0tLS0t"
    "fC0tLS0tLS0tLS0tLS0tLS0tLXwiXQogICAgICAgICAgICBmb3IgZSBpbiBzZWxmLmVycm9yczoKICAgICAgICAgICAgICAgIG5h"
    "bWUgPSBvcy5wYXRoLmJhc2VuYW1lKGVbImZpbGUiXSkucmVwbGFjZSgifCIsICLvvI8iKQogICAgICAgICAgICAgICAgTC5hcHBl"
    "bmQoZiJ8IHtuYW1lfSB8IHtlWydlcnJvciddLnJlcGxhY2UoJ3wnLCAn77yPJyl9IHwiKQogICAgICAgICAgICBMICs9IFsiIiwg"
    "Ii0tLSIsICIiLAogICAgICAgICAgICAgICAgICAiIyMjIOyekOyjvCDrgpjsmKTripQg6rK97JqwIiwgIiIsCiAgICAgICAgICAg"
    "ICAgICAgICIqKuyYmyDsmKTtlLzsiqQg7ZiV7IudKGAueGxzYCBgLnBwdGAgYC5kb2NgKSoqIiwKICAgICAgICAgICAgICAgICAg"
    "Iu2VtOuLuSDtlITroZzqt7jrnqjsl5DshJwg7Je07Ja0ICoq64uk66W4IOydtOumhOycvOuhnCDsoIDsnqUqKiDihpIgIgogICAg"
    "ICAgICAgICAgICAgICAiYC54bHN4YCBgLnBwdHhgIGAuZG9jeGAg66GcIOuwlOq+vCDrkqQg64uk7IucIOuzgO2ZmO2VmOyEuOya"
    "lC4iLAogICAgICAgICAgICAgICAgICAi7ZmV7J6l7J6Q66eMIOuwlOq/lCDsk7Qg7YyM7J2864+EIOqwmeydgCDsmKTrpZjqsIAg"
    "64Kp64uI64ukLiIsCiAgICAgICAgICAgICAgICAgICIiLAogICAgICAgICAgICAgICAgICAiKirquIDsnpDqsIAg7JeG64qUIFBE"
    "RioqIiwKICAgICAgICAgICAgICAgICAgIuyiheydtOulvCDsiqTsupTtlZjqsbDrgpgg7IKs7KeE7Jy866GcIOunjOuToCDrrLjs"
    "hJzsnoXri4jri6QuICIKICAgICAgICAgICAgICAgICAgIuydtCDrj4TqtazripQgKirquIDsnpAg7J247IudKE9DUinsnYQg7ZWY"
    "7KeAIOyViuycvOuvgOuhnCoqIOuzgO2ZmO2VoCDsiJgg7JeG7Iq164uI64ukLiIsCiAgICAgICAgICAgICAgICAgICLsm5Drs7gg"
    "66y47IScIO2MjOydvOydtCDsnojsnLzrqbQg6re46rKD7J2EIOuzgO2ZmO2VmOyEuOyalC4iXQogICAgICAgICAgICB3aXRoIGlv"
    "Lm9wZW4ob3MucGF0aC5qb2luKHNlbGYub3V0LCAiX+yYpOulmC5tZCIpLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAg"
    "ICAgICAgICAgICAgICBmLndyaXRlKCJcbiIuam9pbihMKSkK"
  ),
  "pkems_gdrive.py": (
    "IyAtKi0gY29kaW5nOiB1dGYtOCAtKi0KIiIiClBLRU1TIOq1rOq4gCDrrLjshJwg6rCA7KC47Jik6riwCj09PT09PT09PT09PT09"
    "PT09PT09PT09PT0K6rWs6riAIOusuOyEnMK37Iuc7Yq4wrfsiqzrnbzsnbTrk5zripQgJ+uCtCDsu7Ttk6jthLDsl5Ag7Iuk7LK0"
    "6rCAIOyXhuuKlCcg7Jio65287J24IOusuOyEnOudvOyEnArtjIzsnbzroZzripQg7J297J2EIOyImCDsl4bri6QuIERyaXZlIEFQ"
    "SSDroZwg64K067O064K06riwKGV4cG9ydCkg7ZW07JW8IO2VnOuLpC4KCuy9lOueqeyXkOyEnCDsk7DripQg6rKD7J2EIOyghOyg"
    "nOuhnCDtlZzri6QgKOuzhOuPhCDsnbjspp0g7ISk7KCVIOyXhuydtCDrs7jsnbgg6rOE7KCV7Jy866GcIOuPmeyekSkuCgogICAg"
    "ZnJvbSBwa2Vtc19nZHJpdmUgaW1wb3J0IEdvb2dsZURvY3MKCiAgICBnID0gR29vZ2xlRG9jcygpICAgICAgICAgICAgICAgICAg"
    "ICAgIyDsnbjspp0KICAgIGcubGlzdF9mb2xkZXIoIjFBYkMuLi4iKSAgICAgICAgICAgICAjIO2PtOuNlCDslYgg6rWs6riAIOus"
    "uOyEnCDrqqnroZ0KICAgIGcuZXhwb3J0X2ZvbGRlcigiMUFiQy4uLiIsICIvY29udGVudC9kcml2ZS9NeURyaXZlL1BLRU1TL+q1"
    "rOq4gOusuOyEnCIpCgpQS0VNUyjqsJzsnbjsp4Dsi53qsr3tl5jqtIDrpqzssrTqs4QpIO2UhOuhnOygne2KuAoiIiIKCmZyb20g"
    "X19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBvcwppbXBvcnQgaW8KaW1wb3J0IHJlCmltcG9ydCBqc29uCmlt"
    "cG9ydCB0aW1lCgojIOq1rOq4gCDrrLjshJwg7KKF66WYIC0+IOyWtOuWpCDtmJXsi53snLzroZwg67Cb7JWE7Jis7KeACiMKIyAg"
    "IOusuOyEnCAgIDog6rWs6riA7J20IOuniO2BrOuLpOyatOycvOuhnCDrsJTroZwg64K067O064K0IOykgOuLpAojICAg7Iuc7Yq4"
    "ICAgOiBjc3Yg66GcIOuwm+ycvOuptCAn7LKrIOyepSfrp4wg7Jio64ukIC0+IHhsc3gg66GcIOuwm+yVhCDrqqjrk6Ag7Iuc7Yq4"
    "66W8IO2RnOuhnCDsmK7quLTri6QKIyAgIOyKrOudvOydtOuTnDogdHh0IOuhnCDrsJvsnLzrqbQg67Cc7ZGc7J6QIOuFuO2KuOqw"
    "gCDruaDsp4Tri6QgLT4gcHB0eCDroZwg67Cb7JWEIOuFuO2KuOq5jOyngCDsmK7quLTri6QKIwojIOyWtOuWpCDqsr3smrDrk6Ag"
    "7LWc7KKFIOqysOqzvOuKlCAubWQg7ZWY64KY66GcIO2GteydvO2VnOuLpC4KWExTWF9NSU1FID0gImFwcGxpY2F0aW9uL3ZuZC5v"
    "cGVueG1sZm9ybWF0cy1vZmZpY2Vkb2N1bWVudC5zcHJlYWRzaGVldG1sLnNoZWV0IgpQUFRYX01JTUUgPSAiYXBwbGljYXRpb24v"
    "dm5kLm9wZW54bWxmb3JtYXRzLW9mZmljZWRvY3VtZW50LnByZXNlbnRhdGlvbm1sLnByZXNlbnRhdGlvbiIKCkVYUE9SVF9BUyA9"
    "IHsKICAgICJhcHBsaWNhdGlvbi92bmQuZ29vZ2xlLWFwcHMuZG9jdW1lbnQiOiAgICAgKCJ0ZXh0L21hcmtkb3duIiwgIi5tZCIs"
    "ICLqtazquIDrrLjshJwiKSwKICAgICJhcHBsaWNhdGlvbi92bmQuZ29vZ2xlLWFwcHMuc3ByZWFkc2hlZXQiOiAgKFhMU1hfTUlN"
    "RSwgIi5tZCIsICLqtazquIDsi5ztirgiKSwKICAgICJhcHBsaWNhdGlvbi92bmQuZ29vZ2xlLWFwcHMucHJlc2VudGF0aW9uIjog"
    "KFBQVFhfTUlNRSwgIi5tZCIsICLqtazquIDsiqzrnbzsnbTrk5wiKSwKfQoKRk9MREVSX01JTUUgPSAiYXBwbGljYXRpb24vdm5k"
    "Lmdvb2dsZS1hcHBzLmZvbGRlciIKCl9JTlZBTElEID0gcmUuY29tcGlsZShyJ1tcXC86Kj8iPD58XScpCgoKZGVmIHNhZmVfbmFt"
    "ZShuYW1lOiBzdHIsIG1heGxlbjogaW50ID0gOTApIC0+IHN0cjoKICAgIHJldHVybiAoX0lOVkFMSUQuc3ViKCJfIiwgbmFtZSku"
    "c3RyaXAoKVs6bWF4bGVuXS5yc3RyaXAoIiAuIikpIG9yICLrrLTsoJwiCgoKZGVmIGZvbGRlcl9pZF9mcm9tKHRleHQ6IHN0cikg"
    "LT4gc3RyIHwgTm9uZToKICAgICIiIu2PtOuNlCDrp4Htgawg65iQ64qUIElEIOusuOyekOyXtOyXkOyEnCBJROunjCDrvZHslYTr"
    "grjri6QuIiIiCiAgICB0ZXh0ID0gKHRleHQgb3IgIiIpLnN0cmlwKCkKICAgIG0gPSByZS5zZWFyY2gociIvZm9sZGVycy8oW0Et"
    "WmEtejAtOV8tXXsxMCx9KSIsIHRleHQpCiAgICBpZiBtOgogICAgICAgIHJldHVybiBtLmdyb3VwKDEpCiAgICBtID0gcmUubWF0"
    "Y2gociJeKFtBLVphLXowLTlfLV17MTAsfSkkIiwgdGV4dCkKICAgIHJldHVybiBtLmdyb3VwKDEpIGlmIG0gZWxzZSBOb25lCgoK"
    "Y2xhc3MgR29vZ2xlRG9jczoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCB2ZXJib3NlOiBib29sID0gVHJ1ZSk6CiAgICAgICAgc2Vs"
    "Zi52ZXJib3NlID0gdmVyYm9zZQogICAgICAgIHNlbGYuc3ZjID0gc2VsZi5fY29ubmVjdCgpCgogICAgZGVmIGxvZyhzZWxmLCAq"
    "YSk6CiAgICAgICAgaWYgc2VsZi52ZXJib3NlOgogICAgICAgICAgICBwcmludCgqYSwgZmx1c2g9VHJ1ZSkKCiAgICBkZWYgX2Nv"
    "bm5lY3Qoc2VsZik6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBmcm9tIGdvb2dsZS5jb2xhYiBpbXBvcnQgYXV0aAogICAgICAg"
    "ICAgICBhdXRoLmF1dGhlbnRpY2F0ZV91c2VyKCkKICAgICAgICBleGNlcHQgSW1wb3J0RXJyb3I6CiAgICAgICAgICAgIHBhc3Mg"
    "ICAgICAgICAgICAgICAgICAgICAgICMg7L2U656p7J20IOyVhOuLiOuptCDquLDrs7gg7J6Q6rKp7Kad66qF7J2EIOyCrOyaqQog"
    "ICAgICAgIGZyb20gZ29vZ2xlYXBpY2xpZW50LmRpc2NvdmVyeSBpbXBvcnQgYnVpbGQKICAgICAgICByZXR1cm4gYnVpbGQoImRy"
    "aXZlIiwgInYzIikKCiAgICAjIOKUgOKUgCDtj7TrjZQg7JWIIO2VreuqqSDrgpjsl7QgKO2VmOychCDtj7TrjZTquYzsp4ApCiAg"
    "ICBkZWYgd2FsayhzZWxmLCBmb2xkZXJfaWQ6IHN0ciwgcmVjdXJzaXZlOiBib29sID0gVHJ1ZSwKICAgICAgICAgICAgIF9wcmVm"
    "aXg6IHN0ciA9ICIiKSAtPiBsaXN0W2RpY3RdOgogICAgICAgIGl0ZW1zLCB0b2tlbiA9IFtdLCBOb25lCiAgICAgICAgd2hpbGUg"
    "VHJ1ZToKICAgICAgICAgICAgcmVzcCA9IHNlbGYuc3ZjLmZpbGVzKCkubGlzdCgKICAgICAgICAgICAgICAgIHE9ZiIne2ZvbGRl"
    "cl9pZH0nIGluIHBhcmVudHMgYW5kIHRyYXNoZWQgPSBmYWxzZSIsCiAgICAgICAgICAgICAgICBmaWVsZHM9Im5leHRQYWdlVG9r"
    "ZW4sIGZpbGVzKGlkLG5hbWUsbWltZVR5cGUsbW9kaWZpZWRUaW1lKSIsCiAgICAgICAgICAgICAgICBwYWdlU2l6ZT0yMDAsIHBh"
    "Z2VUb2tlbj10b2tlbiwKICAgICAgICAgICAgICAgIHN1cHBvcnRzQWxsRHJpdmVzPVRydWUsIGluY2x1ZGVJdGVtc0Zyb21BbGxE"
    "cml2ZXM9VHJ1ZSwKICAgICAgICAgICAgKS5leGVjdXRlKCkKICAgICAgICAgICAgZm9yIGYgaW4gcmVzcC5nZXQoImZpbGVzIiwg"
    "W10pOgogICAgICAgICAgICAgICAgZlsicGF0aCJdID0gb3MucGF0aC5qb2luKF9wcmVmaXgsIHNhZmVfbmFtZShmWyJuYW1lIl0p"
    "KQogICAgICAgICAgICAgICAgaWYgZlsibWltZVR5cGUiXSA9PSBGT0xERVJfTUlNRToKICAgICAgICAgICAgICAgICAgICBpZiBy"
    "ZWN1cnNpdmU6CiAgICAgICAgICAgICAgICAgICAgICAgIGl0ZW1zICs9IHNlbGYud2FsayhmWyJpZCJdLCBUcnVlLCBmWyJwYXRo"
    "Il0pCiAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgIGl0ZW1zLmFwcGVuZChmKQogICAgICAgICAgICB0"
    "b2tlbiA9IHJlc3AuZ2V0KCJuZXh0UGFnZVRva2VuIikKICAgICAgICAgICAgaWYgbm90IHRva2VuOgogICAgICAgICAgICAgICAg"
    "YnJlYWsKICAgICAgICByZXR1cm4gaXRlbXMKCiAgICAjIOKUgOKUgCDrqqnroZ0g67O06riwCiAgICBkZWYgbGlzdF9mb2xkZXIo"
    "c2VsZiwgZm9sZGVyX29yX2xpbms6IHN0ciwgcmVjdXJzaXZlOiBib29sID0gVHJ1ZSkgLT4gbGlzdFtkaWN0XToKICAgICAgICBm"
    "aWQgPSBmb2xkZXJfaWRfZnJvbShmb2xkZXJfb3JfbGluaykKICAgICAgICBpZiBub3QgZmlkOgogICAgICAgICAgICBzZWxmLmxv"
    "Zygi7Y+0642UIOunge2BrCDrmJDripQgSUQg7ZiV7Iud7J20IOyVhOuLmeuLiOuLpC4iKQogICAgICAgICAgICByZXR1cm4gW10K"
    "ICAgICAgICBmaWxlcyA9IHNlbGYud2FsayhmaWQsIHJlY3Vyc2l2ZSkKICAgICAgICBnb29nbGUgPSBbZiBmb3IgZiBpbiBmaWxl"
    "cyBpZiBmWyJtaW1lVHlwZSJdIGluIEVYUE9SVF9BU10KICAgICAgICBvdGhlciA9IFtmIGZvciBmIGluIGZpbGVzIGlmIGZbIm1p"
    "bWVUeXBlIl0gbm90IGluIEVYUE9SVF9BU10KCiAgICAgICAgc2VsZi5sb2coZiLsoITssrQge2xlbihmaWxlcyl96rCcIikKICAg"
    "ICAgICBzZWxmLmxvZyhmIiAg6rCA7KC47JisIOyImCDsnojripQg6rWs6riAIOusuOyEnCA6IHtsZW4oZ29vZ2xlKX3qsJwiKQog"
    "ICAgICAgIGNvdW50cyA9IHt9CiAgICAgICAgZm9yIGYgaW4gZ29vZ2xlOgogICAgICAgICAgICBrID0gRVhQT1JUX0FTW2ZbIm1p"
    "bWVUeXBlIl1dWzJdCiAgICAgICAgICAgIGNvdW50c1trXSA9IGNvdW50cy5nZXQoaywgMCkgKyAxCiAgICAgICAgZm9yIGssIG4g"
    "aW4gY291bnRzLml0ZW1zKCk6CiAgICAgICAgICAgIHNlbGYubG9nKGYiICAgICB7azoxMn0ge2596rCcIikKICAgICAgICBzZWxm"
    "LmxvZyhmIiAg7J2867CYIO2MjOydvCjrs4Trj4Qg67OA7ZmYIO2VhOyalCkgOiB7bGVuKG90aGVyKX3qsJwiKQogICAgICAgIHJl"
    "dHVybiBnb29nbGUKCiAgICAjIOKUgOKUgCDtlZwg6rCcIOuCtOuztOuCtOq4sCAo66y07JeH7J2065OgIC5tZCDroZwg66eM65Og"
    "64ukKQogICAgZGVmIGV4cG9ydF9vbmUoc2VsZiwgZmlsZV9pZDogc3RyLCBtaW1lOiBzdHIsIGRzdF9ub2V4dDogc3RyKSAtPiBz"
    "dHIgfCBOb25lOgogICAgICAgIHRhcmdldCwgZXh0LCBfa2luZCA9IEVYUE9SVF9BU1ttaW1lXQoKICAgICAgICB0cnk6CiAgICAg"
    "ICAgICAgIGRhdGEgPSBzZWxmLnN2Yy5maWxlcygpLmV4cG9ydChmaWxlSWQ9ZmlsZV9pZCwgbWltZVR5cGU9dGFyZ2V0KS5leGVj"
    "dXRlKCkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAjIOyalOyyre2VnCDtmJXsi53snYQg7KeA7JuQ7ZWY"
    "7KeAIOyViuycvOuptCDsnbzrsJgg7YWN7Iqk7Yq466GcIO2bhO2HtAogICAgICAgICAgICBkYXRhID0gc2VsZi5zdmMuZmlsZXMo"
    "KS5leHBvcnQoZmlsZUlkPWZpbGVfaWQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtaW1lVHlw"
    "ZT0idGV4dC9wbGFpbiIpLmV4ZWN1dGUoKQogICAgICAgICAgICB0YXJnZXQgPSAidGV4dC9wbGFpbiIKCiAgICAgICAgaWYgbm90"
    "IGlzaW5zdGFuY2UoZGF0YSwgYnl0ZXMpOgogICAgICAgICAgICBkYXRhID0gZGF0YS5lbmNvZGUoInV0Zi04IikKCiAgICAgICAg"
    "cGF0aCA9IGRzdF9ub2V4dCArIGV4dAogICAgICAgIG9zLm1ha2VkaXJzKG9zLnBhdGguZGlybmFtZShwYXRoKSwgZXhpc3Rfb2s9"
    "VHJ1ZSkKCiAgICAgICAgaWYgdGFyZ2V0IGluIChYTFNYX01JTUUsIFBQVFhfTUlNRSk6CiAgICAgICAgICAgIGJvZHkgPSBzZWxm"
    "Ll9vZmZpY2VfdG9fbWQoZGF0YSwgdGFyZ2V0KQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGJvZHkgPSBkYXRhLmRlY29kZSgi"
    "dXRmLTgiLCAiaWdub3JlIikKCiAgICAgICAgd2l0aCBpby5vcGVuKHBhdGgsICJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoK"
    "ICAgICAgICAgICAgZi53cml0ZShib2R5KQogICAgICAgIHJldHVybiBwYXRoCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIF9v"
    "ZmZpY2VfdG9fbWQoZGF0YTogYnl0ZXMsIHRhcmdldDogc3RyKSAtPiBzdHI6CiAgICAgICAgIiIieGxzeC9wcHR4IOybkOuzuOyd"
    "hCDsnbTrr7gg6rKA7Kad65CcIOydveq4sCDrqqjrk4jroZwg66eI7YGs64uk7Jq07Jy866GcIOyYruq4tOuLpC4iIiIKICAgICAg"
    "ICBpbXBvcnQgdGVtcGZpbGUKICAgICAgICBmcm9tIHBrZW1zX3JlYWRlcnMgaW1wb3J0IHJlYWRfeGxzeCwgcmVhZF9wcHR4Cgog"
    "ICAgICAgIHN1ZmZpeCA9ICIueGxzeCIgaWYgdGFyZ2V0ID09IFhMU1hfTUlNRSBlbHNlICIucHB0eCIKICAgICAgICB0bXAgPSB0"
    "ZW1wZmlsZS5OYW1lZFRlbXBvcmFyeUZpbGUoc3VmZml4PXN1ZmZpeCwgZGVsZXRlPUZhbHNlKQogICAgICAgIHRyeToKICAgICAg"
    "ICAgICAgdG1wLndyaXRlKGRhdGEpCiAgICAgICAgICAgIHRtcC5jbG9zZSgpCiAgICAgICAgICAgIHJlcyA9IHJlYWRfeGxzeCh0"
    "bXAubmFtZSkgaWYgc3VmZml4ID09ICIueGxzeCIgZWxzZSByZWFkX3BwdHgodG1wLm5hbWUpCiAgICAgICAgICAgIHJldHVybiBy"
    "ZXMudGV4dCBpZiByZXMub2sgZWxzZSBmIj4g64K07Jqp7J2EIOydveyngCDrqrvtlojsirXri4jri6Q6IHtyZXMuZXJyb3J9Igog"
    "ICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIG9zLnVubGluayh0bXAubmFtZSkKICAgICAg"
    "ICAgICAgZXhjZXB0IE9TRXJyb3I6CiAgICAgICAgICAgICAgICBwYXNzCgogICAgIyDilIDilIAg7Y+0642UIO2GteynuOuhnCDr"
    "grTrs7TrgrTquLAKICAgIGRlZiBleHBvcnRfZm9sZGVyKHNlbGYsIGZvbGRlcl9vcl9saW5rOiBzdHIsIG91dF9kaXI6IHN0ciwK"
    "ICAgICAgICAgICAgICAgICAgICAgIHJlY3Vyc2l2ZTogYm9vbCA9IFRydWUsIHNraXBfZXhpc3Rpbmc6IGJvb2wgPSBUcnVlKSAt"
    "PiBkaWN0OgogICAgICAgIHQwID0gdGltZS50aW1lKCkKICAgICAgICBmaWxlcyA9IHNlbGYubGlzdF9mb2xkZXIoZm9sZGVyX29y"
    "X2xpbmssIHJlY3Vyc2l2ZSkKICAgICAgICBpZiBub3QgZmlsZXM6CiAgICAgICAgICAgICMg7YKk66W8IOu5oOucqOumrOuptCDq"
    "srDqs7zrpbwg67Cb7JWEIOyTsOuKlCDsqr3sl5DshJwg7Jik66WY6rCAIOuCnOuLpAogICAgICAgICAgICByZXR1cm4geyJkb25l"
    "IjogMCwgInNraXBwZWQiOiAwLCAiZmFpbGVkIjogMH0KCiAgICAgICAgc2VsZi5sb2coZiJcbntvdXRfZGlyfSDroZwg6rCA7KC4"
    "7Ji164uI64ukLlxuIikKICAgICAgICBvcy5tYWtlZGlycyhvdXRfZGlyLCBleGlzdF9vaz1UcnVlKQogICAgICAgIGRvbmUgPSBz"
    "a2lwcGVkID0gZmFpbGVkID0gMAogICAgICAgIHJlY29yZHMsIGVycm9ycyA9IFtdLCBbXQoKICAgICAgICBmb3IgaSwgZiBpbiBl"
    "bnVtZXJhdGUoZmlsZXMsIDEpOgogICAgICAgICAgICBraW5kID0gRVhQT1JUX0FTW2ZbIm1pbWVUeXBlIl1dWzJdCiAgICAgICAg"
    "ICAgIGRzdF9ub2V4dCA9IG9zLnBhdGguam9pbihvdXRfZGlyLCBmWyJwYXRoIl0pCiAgICAgICAgICAgIGd1ZXNzID0gZHN0X25v"
    "ZXh0ICsgRVhQT1JUX0FTW2ZbIm1pbWVUeXBlIl1dWzFdCiAgICAgICAgICAgIGlmIHNraXBfZXhpc3RpbmcgYW5kIG9zLnBhdGgu"
    "ZXhpc3RzKGd1ZXNzKToKICAgICAgICAgICAgICAgIHNraXBwZWQgKz0gMQogICAgICAgICAgICAgICAgY29udGludWUKICAgICAg"
    "ICAgICAgdHJ5OgogICAgICAgICAgICAgICAgcGF0aCA9IHNlbGYuZXhwb3J0X29uZShmWyJpZCJdLCBmWyJtaW1lVHlwZSJdLCBk"
    "c3Rfbm9leHQpCiAgICAgICAgICAgICAgICBzZWxmLl9hZGRfZnJvbnRfbWF0dGVyKHBhdGgsIGYsIGtpbmQpCiAgICAgICAgICAg"
    "ICAgICByZWNvcmRzLmFwcGVuZCh7InRpdGxlIjogZlsibmFtZSJdLCAia2luZCI6IGtpbmQsCiAgICAgICAgICAgICAgICAgICAg"
    "ICAgICAgICAgICAgImZpbGUiOiBvcy5wYXRoLnJlbHBhdGgocGF0aCwgb3V0X2RpcikucmVwbGFjZSgiXFwiLCAiLyIpLAogICAg"
    "ICAgICAgICAgICAgICAgICAgICAgICAgICAgICJkYXRlIjogZi5nZXQoIm1vZGlmaWVkVGltZSIsICIiKVs6MTBdLAogICAgICAg"
    "ICAgICAgICAgICAgICAgICAgICAgICAgICJ1cmwiOiBmImh0dHBzOi8vZHJpdmUuZ29vZ2xlLmNvbS9vcGVuP2lkPXtmWydpZCdd"
    "fSJ9KQogICAgICAgICAgICAgICAgZG9uZSArPSAxCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAg"
    "ICAgICAgIGVycm9ycy5hcHBlbmQoeyJuYW1lIjogZlsibmFtZSJdLCAiZXJyb3IiOiBzdHIoZSlbOjIwMF19KQogICAgICAgICAg"
    "ICAgICAgZmFpbGVkICs9IDEKICAgICAgICAgICAgaWYgZG9uZSBhbmQgZG9uZSAlIDIwID09IDA6CiAgICAgICAgICAgICAgICBz"
    "ZWxmLmxvZyhmIiAg4oCmIHtkb25lfeqwnCAoe2l9L3tsZW4oZmlsZXMpfSkiKQoKICAgICAgICBzZWxmLl93cml0ZV9pbmRleChv"
    "dXRfZGlyLCByZWNvcmRzLCBlcnJvcnMpCiAgICAgICAgc2VjcyA9IGludCh0aW1lLnRpbWUoKSAtIHQwKQogICAgICAgIHNlbGYu"
    "bG9nKGYiXG7smYTro4whIOqwgOyguOyYtCB7ZG9uZX3qsJwgwrcg6rG064SI65yAIHtza2lwcGVkfeqwnCDCtyDsi6TtjKgge2Zh"
    "aWxlZH3qsJwgIgogICAgICAgICAgICAgICAgIGYiwrcge3NlY3MgLy8gNjB967aEIHtzZWNzICUgNjB97LSIIikKICAgICAgICBy"
    "ZXR1cm4geyJkb25lIjogZG9uZSwgInNraXBwZWQiOiBza2lwcGVkLCAiZmFpbGVkIjogZmFpbGVkfQoKICAgICMg4pSA4pSAIOuC"
    "tOuztOuCuCDtjIzsnbwg7JWe7JeQIOygleuztCDrtpnsnbTquLAKICAgIGRlZiBfYWRkX2Zyb250X21hdHRlcihzZWxmLCBwYXRo"
    "OiBzdHIsIGY6IGRpY3QsIGtpbmQ6IHN0cik6CiAgICAgICAgdHJ5OgogICAgICAgICAgICB3aXRoIGlvLm9wZW4ocGF0aCwgZW5j"
    "b2Rpbmc9InV0Zi04IikgYXMgZmg6CiAgICAgICAgICAgICAgICBib2R5ID0gZmgucmVhZCgpCiAgICAgICAgZXhjZXB0IEV4Y2Vw"
    "dGlvbjoKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgdGl0bGUgPSBmWyJuYW1lIl0ucmVwbGFjZSgnIicsICInIikKICAgICAg"
    "ICBoZWFkID0gKCItLS1cbiIKICAgICAgICAgICAgICAgIGYndGl0bGU6ICJ7dGl0bGV9IlxuJwogICAgICAgICAgICAgICAgZidk"
    "YXRlOiB7Zi5nZXQoIm1vZGlmaWVkVGltZSIsIiIpWzoxMF19XG4nCiAgICAgICAgICAgICAgICBmJ2tpbmQ6ICJ7a2luZH0iXG4n"
    "CiAgICAgICAgICAgICAgICBmJ3VybDogaHR0cHM6Ly9kcml2ZS5nb29nbGUuY29tL29wZW4/aWQ9e2ZbImlkIl19XG4nCiAgICAg"
    "ICAgICAgICAgICAiLS0tXG5cbiIKICAgICAgICAgICAgICAgIGYiIyB7dGl0bGV9XG5cbiIpCiAgICAgICAgd2l0aCBpby5vcGVu"
    "KHBhdGgsICJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZmg6CiAgICAgICAgICAgIGZoLndyaXRlKGhlYWQgKyBib2R5KQoKICAg"
    "IGRlZiBfd3JpdGVfaW5kZXgoc2VsZiwgb3V0X2Rpcjogc3RyLCByZWNvcmRzOiBsaXN0LCBlcnJvcnM6IGxpc3QpOgogICAgICAg"
    "IGlmIHJlY29yZHM6CiAgICAgICAgICAgIHdpdGggaW8ub3Blbihvcy5wYXRoLmpvaW4ob3V0X2RpciwgIl9maWxlcy5qc29uIiks"
    "ICJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICAgICAgICAgIGpzb24uZHVtcChyZWNvcmRzLCBmLCBlbnN1cmVf"
    "YXNjaWk9RmFsc2UsIGluZGVudD0xKQogICAgICAgICAgICBMID0gWyIjIPCfk4Qg6rCA7KC47JioIOq1rOq4gCDrrLjshJwiLCAi"
    "IiwgZiLsoITssrQgKip7bGVuKHJlY29yZHMpfeqwnCoqIiwgIiJdCiAgICAgICAgICAgIGZvciByIGluIHNvcnRlZChyZWNvcmRz"
    "LCBrZXk9bGFtYmRhIHg6IHhbImZpbGUiXSk6CiAgICAgICAgICAgICAgICBMLmFwcGVuZChmIi0gW3tyWyd0aXRsZSddfV0oe3Jb"
    "J2ZpbGUnXX0pIMK3IHtyWydraW5kJ119IMK3IHtyWydkYXRlJ119IikKICAgICAgICAgICAgd2l0aCBpby5vcGVuKG9zLnBhdGgu"
    "am9pbihvdXRfZGlyLCAiSU5ERVgubWQiKSwgInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgICAgICAgICAgZi53"
    "cml0ZSgiXG4iLmpvaW4oTCkpCiAgICAgICAgaWYgZXJyb3JzOgogICAgICAgICAgICBMID0gWyIjIOKaoCDqsIDsoLjsmKTsp4Ag"
    "66q77ZWcIOusuOyEnCIsICIiLCAifCDrrLjshJwgfCDsnbTsnKAgfCIsICJ8LS0tLS0tfC0tLS0tLXwiXQogICAgICAgICAgICBm"
    "b3IgZSBpbiBlcnJvcnM6CiAgICAgICAgICAgICAgICBMLmFwcGVuZChmInwge2VbJ25hbWUnXS5yZXBsYWNlKCd8Jywn77yPJyl9"
    "IHwge2VbJ2Vycm9yJ119IHwiKQogICAgICAgICAgICB3aXRoIGlvLm9wZW4ob3MucGF0aC5qb2luKG91dF9kaXIsICJf7Jik66WY"
    "Lm1kIiksICJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICAgICAgICAgIGYud3JpdGUoIlxuIi5qb2luKEwpKQo="
  ),
}

for _name, _b64 in _ENGINES.items():
    pathlib.Path(_name).write_bytes(base64.b64decode(_b64))
sys.path.insert(0, ".")

import pkems_converter, pkems_readers, pkems_privacy, pkems_folder
for _m in (pkems_converter, pkems_readers, pkems_privacy, pkems_folder):
    importlib.reload(_m)
from pkems_converter import Converter, Settings, inspect
from pkems_readers import read_any, SUPPORTED
from pkems_privacy import PrivacyFilter, Policy, preview as 개인정보_미리보기
from pkems_folder import FolderConverter, FolderSettings

print("엔진 준비 완료!")
print("다룰 수 있는 형식:", " ".join(SUPPORTED))

---
---

# 📝 1부 · 네이버 블로그 백업 PDF 변환

블로그 글이 **한 편씩 따로** 마크다운 파일이 됩니다. (제목·날짜·카테고리·원문주소 포함)

### 미리 준비할 것

1. **블로그를 PDF로 백업**
   블로그 관리 → 글 전체보기 → 인쇄 → 대상을 **'PDF로 저장'**
   (100편 정도씩 나눠 저장하면 안정적입니다)
2. 구글 드라이브에 폴더를 만들고 PDF 넣기

*(블로그가 없으면 이 부는 건너뛰고 2부로 가세요)*

## 1-① PDF가 들어있는 폴더 알려주기

아래 **세 가지 방법 중 아무거나** 하나만 채우면 됩니다.

| 방법 | 예시 |
|------|------|
| 폴더 **링크** 붙여넣기 | `https://drive.google.com/drive/folders/1AbC...` |
| 폴더 **ID**만 붙여넣기 | `1AbCdEfGhIjK...` |
| 내 드라이브 안 **경로** | `블로그백업` 또는 `기록/블로그백업` |

In [ ]:
#@title ▶ 폴더 지정하기 { display-mode: "form" }
#@markdown ### 폴더 링크 또는 ID (둘 중 하나, 없으면 비워두세요)
드라이브_링크_또는_ID = ""  #@param {type:"string"}
#@markdown ### 또는, 내 드라이브 안의 폴더 경로
내_드라이브_경로 = "블로그백업"  #@param {type:"string"}

import os, re

MYDRIVE = "/content/drive/MyDrive"


def _folder_path_from_id(folder_id):
    """드라이브 폴더 ID -> 마운트된 실제 경로"""
    from google.colab import auth
    from googleapiclient.discovery import build
    auth.authenticate_user()
    svc = build("drive", "v3")
    parts = []
    cur = folder_id
    for _ in range(20):
        info = svc.files().get(fileId=cur, fields="id,name,parents").execute()
        parts.append(info["name"])
        parents = info.get("parents")
        if not parents:
            break
        cur = parents[0]
    parts.reverse()
    # 최상위(내 드라이브) 이름은 버리고 이어붙인다
    return os.path.join(MYDRIVE, *parts[1:]) if len(parts) > 1 else MYDRIVE


PDF_DIR = None
raw = 드라이브_링크_또는_ID.strip()

if raw:
    m = re.search(r"/folders/([A-Za-z0-9_-]{10,})", raw) or re.match(r"^([A-Za-z0-9_-]{10,})$", raw)
    if not m:
        print("링크/ID 형식을 알아보지 못했습니다. 폴더 주소를 그대로 붙여넣어 보세요.")
    else:
        try:
            PDF_DIR = _folder_path_from_id(m.group(1))
            print(f"폴더를 찾았습니다: {PDF_DIR}")
        except Exception as e:
            print(f"ID로 찾기 실패({e}). 아래 '경로' 방식을 써주세요.")

if PDF_DIR is None:
    PDF_DIR = os.path.join(MYDRIVE, 내_드라이브_경로.strip().strip("/"))

print()
if os.path.isdir(PDF_DIR):
    pdfs = sorted(n for n in os.listdir(PDF_DIR) if n.lower().endswith(".pdf"))
    print(f"경로 : {PDF_DIR}")
    print(f"PDF  : {len(pdfs)}개 발견")
    for n in pdfs[:15]:
        mb = os.path.getsize(os.path.join(PDF_DIR, n)) / 1024 / 1024
        print(f"   - {n}  ({mb:.1f} MB)")
    if len(pdfs) > 15:
        print(f"   … 외 {len(pdfs)-15}개")
    if not pdfs:
        print("\n⚠ 이 폴더에 PDF가 없습니다. 폴더를 다시 확인해주세요.")
else:
    print(f"⚠ 폴더를 찾을 수 없습니다: {PDF_DIR}")
    print("   왼쪽 파일 탐색기(📁)에서 실제 폴더 이름을 확인해보세요.")

## 1-② 미리 확인하기 (권장)

변환하기 전에, PDF가 올바른 형식인지 **미리 훑어봅니다.**
글이 몇 편 들어있는지 여기서 확인할 수 있어요.

In [ ]:
#@title ▶ 미리 확인하기 { display-mode: "form" }
import os

pdfs = sorted(n for n in os.listdir(PDF_DIR) if n.lower().endswith(".pdf"))
총합 = 0
for n in pdfs:
    r = inspect(os.path.join(PDF_DIR, n))
    총합 += r["posts"]
    print("-" * 46)
print(f"\n예상 변환 결과: 전체 약 {총합}편")

## 1-③ 변환 실행

설정을 확인하고 ▶ 를 누르세요. PDF 양에 따라 몇 분 걸립니다.

In [ ]:
#@title ▶ 변환 시작 { display-mode: "form" }
#@markdown ### 결과를 저장할 폴더 이름 (PDF 폴더 안에 생깁니다)
저장폴더 = "md"  #@param {type:"string"}
#@markdown ### 본문 사진도 함께 저장할까요?
사진_저장 = True  #@param {type:"boolean"}
#@markdown ### 이미 변환한 글은 건너뛸까요? (다시 돌려도 안전)
중복_건너뛰기 = True  #@param {type:"boolean"}
#@markdown ### 파일 이름 형식
파일이름형식 = "{date}_{title}"  #@param ["{date}_{title}", "{title}", "{date}"]

import os

OUT_DIR = os.path.join(PDF_DIR, 저장폴더.strip() or "md")

conv = Converter(Settings(
    pdf_dir          = PDF_DIR,
    out_dir          = OUT_DIR,
    extract_images   = 사진_저장,
    skip_existing    = 중복_건너뛰기,
    filename_pattern = 파일이름형식,
))
결과 = conv.run()

print()
print("저장 위치:", OUT_DIR)
print("구글 드라이브에 반영되기까지 잠시 걸릴 수 있습니다.")

## 1-④ 결과 살펴보기

In [ ]:
#@title ▶ 결과 요약 보기 { display-mode: "form" }
import os, io, json, collections

idx_path = os.path.join(OUT_DIR, "_index.json")
with io.open(idx_path, encoding="utf-8") as f:
    idx = json.load(f)

print(f"전체 {len(idx)}편\n")

years = collections.Counter(e["date"][:4] for e in idx)
print("연도별")
for y in sorted(years):
    print(f"   {y} : {years[y]:4}편  " + "█" * min(40, years[y] // 3))

print("\n카테고리 상위 10")
for c, n in collections.Counter(e.get("category", "") for e in idx).most_common(10):
    print(f"   {n:4}편  {c or '(없음)'}")

print(f"\n기간 : {min(e['date'] for e in idx)} ~ {max(e['date'] for e in idx)}")
print(f"목차 : {os.path.join(OUT_DIR, 'INDEX.md')}")

---
---

# 📂 2부 · 문서 폴더 통째로 변환

블로그 PDF 말고도, **폴더 하나를 통째로** 마크다운으로 바꿀 수 있습니다.

| 다루는 형식 | |
|---|---|
| 한글 | `.hwp` `.hwpx` |
| 오피스 | `.docx` `.pptx` `.xlsx` |
| 그 외 | `.pdf` `.html` `.txt` `.csv` |

원래 폴더 구조를 그대로 유지하며, 한 파일이 실패해도 나머지는 계속 진행됩니다.

> ⚠️ **개인정보 주의** — 업무 문서에는 이름·연락처·계좌 같은 정보가 들어있을 수 있습니다.
> 변환 결과를 웹에 올릴 때는 **반드시 선별**하세요.

In [ ]:
#@title ▶ 폴더 훑어보기 (변환 없이 현황만) { display-mode: "form" }
#@markdown ### 변환할 폴더 (내 드라이브 안 경로)
문서폴더 = "01_학교"  #@param {type:"string"}
#@markdown ### 결과를 저장할 폴더
결과폴더 = "PKEMS/변환결과"  #@param {type:"string"}

import os
SRC_DIR = os.path.join("/content/drive/MyDrive", 문서폴더.strip().strip("/"))
DST_DIR = os.path.join("/content/drive/MyDrive", 결과폴더.strip().strip("/"))

if not os.path.isdir(SRC_DIR):
    print(f"⚠ 폴더를 찾을 수 없습니다: {SRC_DIR}")
else:
    fc = FolderConverter(FolderSettings(src_dir=SRC_DIR, out_dir=DST_DIR))
    fc.scan()

## 2-② 개인정보를 어떻게 가릴지 정하기

종류마다 처리 방식을 고를 수 있습니다.

| 방식 | 뜻 | 예시 |
|------|-----|------|
| **부분가림** | 일부만 남김 | 이운희 → `이**` · 010-1234-5678 → `010-****-****` |
| **가림** | 전부 가림 | 900101-1234567 → `******-*******` |
| **삭제** | 아예 지움 | (빈칸) |
| **그대로** | 건드리지 않음 | 이운희 |

In [ ]:
#@title ▶ 개인정보 설정 { display-mode: "form" }
#@markdown ### 개인정보를 가릴까요?
개인정보_가리기 = True  #@param {type:"boolean"}
#@markdown ---
#@markdown ### 종류별 처리 방식
주민등록번호 = "가림"      #@param ["가림", "부분가림", "삭제", "그대로"]
전화번호 = "부분가림"      #@param ["부분가림", "가림", "삭제", "그대로"]
이름 = "부분가림"          #@param ["부분가림", "가림", "삭제", "그대로"]
계좌번호 = "가림"          #@param ["가림", "부분가림", "삭제", "그대로"]
카드번호 = "가림"          #@param ["가림", "부분가림", "삭제", "그대로"]
이메일 = "부분가림"        #@param ["부분가림", "가림", "삭제", "그대로"]
주소 = "부분가림"          #@param ["부분가림", "가림", "삭제", "그대로"]
생년월일 = "부분가림"      #@param ["부분가림", "가림", "삭제", "그대로"]
차량번호 = "부분가림"      #@param ["부분가림", "가림", "삭제", "그대로"]
#@markdown ---
#@markdown ### 이름 찾는 강도
#@markdown `라벨만`=성명·담당자 옆의 이름만 · `보통`=+문서 안 반복 등장(권장) · `적극적`=+성씨 추정(오탐 늘어남)
이름_탐지강도 = "보통"  #@param ["보통", "라벨만", "적극적"]
#@markdown ### 보고서에 가리기 전 원본을 남길까요?
#@markdown 켜면 무엇이 바뀌었는지 대조할 수 있지만, **보고서 자체가 개인정보 덩어리**가 됩니다.
보고서_원본표시 = True  #@param {type:"boolean"}

정책 = Policy(
    주민등록번호=주민등록번호, 전화번호=전화번호, 이름=이름,
    계좌번호=계좌번호, 카드번호=카드번호, 이메일=이메일,
    주소=주소, 생년월일=생년월일, 차량번호=차량번호,
    이름_탐지강도=이름_탐지강도,
)
print("설정 완료")
for k, v in vars(정책).items():
    print(f"   {k:12} {v}")

### 미리보기 — 파일 하나로 시험해보기 (권장)

전체를 돌리기 전에 **파일 한 개**로 어떻게 가려지는지 확인해보세요.

In [ ]:
#@title ▶ 파일 하나로 미리보기 { display-mode: "form" }
#@markdown ### 확인할 파일 (내 드라이브 안 경로, 비우면 폴더에서 자동 선택)
확인할_파일 = ""  #@param {type:"string"}

import os

경로 = os.path.join("/content/drive/MyDrive", 확인할_파일.strip().strip("/")) \
       if 확인할_파일.strip() else None

if 경로 is None:
    fc0 = FolderConverter(FolderSettings(src_dir=SRC_DIR, out_dir=DST_DIR))
    후보 = fc0.collect()
    경로 = 후보[0] if 후보 else None

if not 경로:
    print("확인할 파일을 찾지 못했습니다.")
else:
    print("파일 :", os.path.basename(경로), "\n")
    _r = read_any(경로)
    if not _r.ok:
        print("읽기 실패:", _r.error)
    else:
        개인정보_미리보기(_r.text, 정책)

## 2-③ 폴더 변환 시작

In [ ]:
#@title ▶ 폴더 변환 시작 { display-mode: "form" }
#@markdown ### 먼저 몇 개만 시험해볼까요? (0 = 전부)
시험_개수 = 30  #@param {type:"integer"}
#@markdown ### 이미 변환한 파일은 건너뛸까요?
중복_건너뛰기 = True  #@param {type:"boolean"}
#@markdown ### 원본 폴더 구조를 유지할까요?
폴더구조_유지 = True  #@param {type:"boolean"}

fc = FolderConverter(FolderSettings(
    src_dir        = SRC_DIR,
    out_dir        = DST_DIR,
    skip_existing  = 중복_건너뛰기,
    keep_tree      = 폴더구조_유지,
    개인정보_가리기 = 개인정보_가리기,
    개인정보_정책   = 정책,
    보고서_원본표시 = 보고서_원본표시,
))
결과 = fc.run(limit=(시험_개수 or None))

print()
print("저장 위치   :", DST_DIR)
print("목차        :", os.path.join(DST_DIR, "INDEX.md"))
print("개인정보보고서:", os.path.join(DST_DIR, "_개인정보_보고서.md"))

---
---

# 📄 3부 · 구글 문서·시트·슬라이드 가져오기

구글 문서는 **내 컴퓨터에 실체가 없는 온라인 문서**라서, 파일로는 읽을 수 없습니다.
Drive API 로 **내보내기(export)** 해야 합니다. (구글 문서 → 마크다운, 시트 → CSV)

처음 실행하면 계정 접근 허용을 한 번 더 물어봅니다.

In [ ]:
#@title ▶ ① 구글 문서 목록 보기 { display-mode: "form" }
#@markdown ### 폴더 링크 또는 ID
구글_폴더 = ""  #@param {type:"string"}
#@markdown ### 하위 폴더까지 찾을까요?
하위폴더_포함 = True  #@param {type:"boolean"}

import importlib, pkems_gdrive
importlib.reload(pkems_gdrive)
from pkems_gdrive import GoogleDocs

if not 구글_폴더.strip():
    print("폴더 링크나 ID를 입력해주세요.")
else:
    gd = GoogleDocs()
    문서목록 = gd.list_folder(구글_폴더, recursive=하위폴더_포함)

In [ ]:
#@title ▶ ② 구글 문서 가져오기 { display-mode: "form" }
#@markdown ### 저장할 폴더 (내 드라이브 안 경로)
구글_저장폴더 = "PKEMS/구글문서"  #@param {type:"string"}

import os
G_OUT = os.path.join("/content/drive/MyDrive", 구글_저장폴더.strip().strip("/"))
결과 = gd.export_folder(구글_폴더, G_OUT, recursive=하위폴더_포함)
print()
print("저장 위치 :", G_OUT)

---

### 잘 안 될 때

| 증상 | 해결 |
|------|------|
| 폴더를 찾을 수 없다 | 왼쪽 📁 아이콘 → `drive/MyDrive` 에서 실제 폴더명 확인 |
| 글을 0편 발견 | 네이버 블로그 **인쇄 → PDF 저장** 방식의 백업인지 확인 |
| 중간에 멈춤 | 코랩 연결이 끊긴 것. 다시 ▶ 누르면 **이어서** 진행됩니다 |
| 사진이 너무 많다 | `사진_저장`을 끄고 다시 실행 |

### 다음 단계

변환된 `.md` 파일들은 그대로 **나만의 지식창고**가 됩니다.
Claude·ChatGPT 같은 AI에게 폴더째 물어보거나, 웹 뷰어로 만들어 검색할 수 있습니다.

*PKEMS · 개인지식경험관리체계*